# H-T01 vs H-T02 최종 모델 선정 분석

동일한 validation 40명에서 최고 checkpoint(T01 epoch 268, T02 epoch 275)를 비교합니다.

- 3D connected component: 26-connectivity
- 크기군: `<1 mL`, `1–5 mL`, `5–20 mL`, `≥20 mL`
- 탐지(any): 정답 병변과 예측 병변이 1 voxel 이상 겹침
- 탐지(10%): 정답 병변 voxel의 10% 이상을 예측이 덮음
- 다발성: 정답 component가 2개 이상인 환자
- 결과: 병변/환자별 CSV, 크기별 탐지율·Dice, 다발성 전체 탐지율·누락·FP, paired 비교와 그림

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, nibabel as nib
import matplotlib.pyplot as plt, seaborn as sns
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from IPython.display import display
ROOT=Path('/home/stu03/workspace/model')
DATA_ROOT=Path('/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet')
GT_DIR=DATA_ROOT/'nnUNet_raw/Dataset001_ICHBinary/labelsTr'
OUTPUT_DIR=ROOT/'runs/comparisons/H-T01_H-T02_lesion_final_selection'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
EXPERIMENTS={
'H-T01-E300':ROOT/'runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTuneA04B06SgdE300V1__3d_fullres/fold_0/full_validation/epoch_0268',
'H-T02-E300':ROOT/'runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansTuneA02B08SgdE300V1__3d_fullres/fold_0/full_validation/epoch_0275'}
SIZE_ORDER=['<1 mL','1–5 mL','5–20 mL','≥20 mL']; STRUCT=np.ones((3,3,3),np.uint8)
for name,d in EXPERIMENTS.items(): assert len(list(d.glob('*.nii.gz')))==40,(name,d)
print('입력 예측 40건씩 확인 완료')

In [ ]:
def div(a,b,default=np.nan): return float(a/b) if b else default
def group(v): return '<1 mL' if v<1 else ('1–5 mL' if v<5 else ('5–20 mL' if v<20 else '≥20 mL'))
def voxel_metrics(g,p):
 tp=int((g&p).sum()); fp=int((~g&p).sum()); fn=int((g&~p).sum()); tn=int((~g&~p).sum())
 return {'tp_voxels':tp,'fp_voxels':fp,'fn_voxels':fn,'tn_voxels':tn,'dice':div(2*tp,2*tp+fp+fn,1.),'iou':div(tp,tp+fp+fn,1.),'recall':div(tp,tp+fn,1.),'precision':div(tp,tp+fp,1.),'specificity':div(tn,tn+fp,1.)}
def components(g,p,vox_ml,exp,case):
 gc,ng=ndimage.label(g,STRUCT); pc,npred=ndimage.label(p,STRUCT)
 gs=np.bincount(gc.ravel(),minlength=ng+1); ps=np.bincount(pc.ravel(),minlength=npred+1)
 ov=np.zeros((ng,npred),dtype=np.int64); both=(gc>0)&(pc>0)
 if both.any():
  pairs=np.stack((gc[both]-1,pc[both]-1),1); u,n=np.unique(pairs,axis=0,return_counts=True); ov[u[:,0],u[:,1]]=n
 matched=np.zeros(ng,bool); match_iou=np.zeros(ng)
 if ng and npred:
  union=gs[1:,None]+ps[None,1:]-ov; iou=np.divide(ov,union,out=np.zeros_like(ov,dtype=float),where=union>0)
  rr,cc=linear_sum_assignment(-iou); valid=iou[rr,cc]>0; matched[rr[valid]]=True; match_iou[rr[valid]]=iou[rr[valid],cc[valid]]
 rows=[]
 for i in range(ng):
  j=int(np.argmax(ov[i])) if npred and ov[i].max()>0 else -1; inter=int(ov[i,j]) if j>=0 else 0
  gv=int(gs[i+1]); pv=int(ps[j+1]) if j>=0 else 0; cov=div(inter,gv,0.)
  rows.append({'experiment':exp,'case_id':case,'source':'PhysioNet' if case.startswith('PHYS_') else 'BHSD','gt_lesion_id':i+1,'gt_volume_ml':gv*vox_ml,'size_group':group(gv*vox_ml),'detected_any_overlap':inter>0,'detected_10pct':cov>=.1,'gt_coverage_recall':cov,'best_component_dice':div(2*inter,gv+pv,0.),'best_component_iou':div(inter,gv+pv-inter,0.),'one_to_one_matched':matched[i],'one_to_one_iou':match_iou[i]})
 hit=(ov>0).any(0) if ng and npred else np.zeros(npred,bool); da=sum(x['detected_any_overlap'] for x in rows); ds=sum(x['detected_10pct'] for x in rows)
 patient={'experiment':exp,'case_id':case,'source':'PhysioNet' if case.startswith('PHYS_') else 'BHSD','gt_lesions':ng,'pred_lesions':npred,'detected_lesions_any':da,'detected_lesions_10pct':ds,'missed_lesions_any':ng-da,'missed_lesions_10pct':ng-ds,'fp_pred_components':int(npred-hit.sum()),'lesion_recall_any':div(da,ng,1.),'lesion_recall_10pct':div(ds,ng,1.),'lesion_precision_any':div(hit.sum(),npred,1.),'any_lesion_detected':da>0 if ng else npred==0,'all_lesions_detected_any':da==ng,'all_lesions_detected_10pct':ds==ng,'exact_component_count':ng==npred,'multiplicity':'negative' if ng==0 else ('single' if ng==1 else 'multiple')}
 return rows,patient

In [ ]:
lr,cr=[],[]
for exp,d in EXPERIMENTS.items():
 for pp in sorted(d.glob('*.nii.gz')):
  case=pp.name[:-7]; gp=GT_DIR/pp.name
  gi,pi=nib.load(gp),nib.load(pp); g=np.asarray(gi.dataobj)>0; pred=np.asarray(pi.dataobj)>0
  if g.shape!=pred.shape: raise ValueError(f'{case}: {g.shape}!={pred.shape}')
  vox=abs(np.linalg.det(gi.affine[:3,:3]))/1000.; lesions,patient=components(g,pred,vox,exp,case)
  patient.update(voxel_metrics(g,pred)); patient.update({'gt_volume_ml':g.sum()*vox,'pred_volume_ml':pred.sum()*vox,'absolute_volume_error_ml':abs(int(pred.sum())-int(g.sum()))*vox})
  lr.extend(lesions); cr.append(patient)
lesions=pd.DataFrame(lr); cases=pd.DataFrame(cr); lesions['size_group']=pd.Categorical(lesions.size_group,SIZE_ORDER,ordered=True)
print('case rows:',len(cases),'lesion rows:',len(lesions))
display(cases.groupby('experiment')[['gt_lesions','pred_lesions']].sum())

In [ ]:
size_summary=(lesions.groupby(['experiment','size_group'],observed=False).agg(gt_lesions=('gt_lesion_id','size'),mean_volume_ml=('gt_volume_ml','mean'),detection_rate_any=('detected_any_overlap','mean'),detection_rate_10pct=('detected_10pct','mean'),mean_component_dice=('best_component_dice','mean'),median_component_dice=('best_component_dice','median'),mean_lesion_recall=('gt_coverage_recall','mean'),one_to_one_detection_rate=('one_to_one_matched','mean')).reset_index())
under5=(lesions[lesions.gt_volume_ml<5].groupby('experiment').agg(gt_lesions=('gt_lesion_id','size'),detection_rate_any=('detected_any_overlap','mean'),detection_rate_10pct=('detected_10pct','mean'),mean_component_dice=('best_component_dice','mean'),mean_lesion_recall=('gt_coverage_recall','mean')).reset_index())
print('병변 크기별'); display(size_summary); print('5 mL 미만'); display(under5)
pos=cases[cases.gt_lesions>0]; multi=cases[cases.gt_lesions>=2]
overall=(pos.groupby('experiment').agg(positive_cases=('case_id','size'),macro_dice=('dice','mean'),mean_iou=('iou','mean'),mean_voxel_recall=('recall','mean'),mean_voxel_precision=('precision','mean'),detected_lesions=('detected_lesions_any','sum'),gt_lesions=('gt_lesions','sum'),mean_fp_components_per_case=('fp_pred_components','mean')).reset_index()); overall['lesion_detection_rate_any']=overall.detected_lesions/overall.gt_lesions
def sm(g): return pd.Series({'multiple_cases':len(g),'total_gt_lesions':g.gt_lesions.sum(),'mean_detected_fraction_any':g.lesion_recall_any.mean(),'mean_detected_fraction_10pct':g.lesion_recall_10pct.mean(),'any_lesion_patient_rate':g.any_lesion_detected.mean(),'all_lesions_patient_rate_any':g.all_lesions_detected_any.mean(),'all_lesions_patient_rate_10pct':g.all_lesions_detected_10pct.mean(),'exact_component_count_rate':g.exact_component_count.mean(),'mean_missed_lesions':g.missed_lesions_any.mean(),'mean_fp_components':g.fp_pred_components.mean()})
multiple_summary=multi.groupby('experiment').apply(sm,include_groups=False).reset_index()
print('전체 양성 환자'); display(overall); print('다발성 환자'); display(multiple_summary)
display(multi[['experiment','case_id','gt_lesions','pred_lesions','detected_lesions_any','detected_lesions_10pct','missed_lesions_any','fp_pred_components','all_lesions_detected_any','all_lesions_detected_10pct','dice']].sort_values(['case_id','experiment']))

In [ ]:
paired=cases.pivot(index='case_id',columns='experiment',values=['dice','recall','precision','lesion_recall_any','lesion_recall_10pct','fp_pred_components'])
for m in ['dice','recall','precision','lesion_recall_any','lesion_recall_10pct','fp_pred_components']: paired[(m,'T02_minus_T01')]=paired[(m,'H-T02-E300')]-paired[(m,'H-T01-E300')]
paired.columns=[f'{a}__{b}' for a,b in paired.columns]; paired=paired.reset_index()
paired_summary=pd.DataFrame([{'metric':m,'T01_mean':cases[cases.experiment=='H-T01-E300'][m].mean(),'T02_mean':cases[cases.experiment=='H-T02-E300'][m].mean(),'T02_minus_T01_mean':paired[f'{m}__T02_minus_T01'].mean(),'T02_better_cases':(paired[f'{m}__T02_minus_T01']>0).sum(),'T01_better_cases':(paired[f'{m}__T02_minus_T01']<0).sum()} for m in ['dice','recall','precision','lesion_recall_any','lesion_recall_10pct']])
display(paired_summary)

In [ ]:
sns.set_theme(style='whitegrid'); fig,ax=plt.subplots(2,2,figsize=(14,10))
sns.barplot(data=size_summary,x='size_group',y='detection_rate_any',hue='experiment',ax=ax[0,0]); ax[0,0].set_ylim(0,1.05); ax[0,0].set_title('Detection: any overlap')
sns.barplot(data=size_summary,x='size_group',y='detection_rate_10pct',hue='experiment',ax=ax[0,1]); ax[0,1].set_ylim(0,1.05); ax[0,1].set_title('Detection: ≥10% coverage')
sns.barplot(data=size_summary,x='size_group',y='mean_component_dice',hue='experiment',ax=ax[1,0]); ax[1,0].set_ylim(0,1.05); ax[1,0].set_title('Mean component Dice')
sns.boxplot(data=pos,x='experiment',y='dice',ax=ax[1,1]); sns.stripplot(data=pos,x='experiment',y='dice',color='black',alpha=.4,ax=ax[1,1]); ax[1,1].set_title('Positive-case Dice')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'lesion_size_comparison.png',dpi=180,bbox_inches='tight'); plt.show()
mp=multiple_summary.melt(id_vars='experiment',value_vars=['mean_detected_fraction_any','all_lesions_patient_rate_any','all_lesions_patient_rate_10pct'],var_name='metric',value_name='value')
fig,ax=plt.subplots(1,3,figsize=(16,4.5)); sns.barplot(data=mp,x='metric',y='value',hue='experiment',ax=ax[0]); ax[0].set_ylim(0,1.05); ax[0].tick_params(axis='x',rotation=25)
sns.barplot(data=multiple_summary,x='experiment',y='mean_missed_lesions',ax=ax[1]); sns.barplot(data=multiple_summary,x='experiment',y='mean_fp_components',ax=ax[2]); ax[1].set_title('Missed lesions/case'); ax[2].set_title('FP components/case')
fig.tight_layout(); fig.savefig(OUTPUT_DIR/'multiple_lesion_comparison.png',dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
lesions.to_csv(OUTPUT_DIR/'per_lesion_metrics.csv',index=False); cases.to_csv(OUTPUT_DIR/'per_case_metrics.csv',index=False); size_summary.to_csv(OUTPUT_DIR/'lesion_size_summary.csv',index=False); under5.to_csv(OUTPUT_DIR/'under5ml_summary.csv',index=False); overall.to_csv(OUTPUT_DIR/'overall_summary.csv',index=False); multiple_summary.to_csv(OUTPUT_DIR/'multiple_lesion_summary.csv',index=False); paired.to_csv(OUTPUT_DIR/'paired_case_comparison.csv',index=False); paired_summary.to_csv(OUTPUT_DIR/'paired_summary.csv',index=False)
selection=overall[['experiment','macro_dice','mean_voxel_recall','mean_voxel_precision','lesion_detection_rate_any','mean_fp_components_per_case']].merge(under5[['experiment','detection_rate_any','detection_rate_10pct','mean_component_dice']].rename(columns={'detection_rate_any':'under5_detection_any','detection_rate_10pct':'under5_detection_10pct','mean_component_dice':'under5_component_dice'}),on='experiment').merge(multiple_summary[['experiment','all_lesions_patient_rate_any','all_lesions_patient_rate_10pct','mean_missed_lesions','mean_fp_components']],on='experiment')
selection.to_csv(OUTPUT_DIR/'final_selection_table.csv',index=False); display(selection)
print('저장 완료:',OUTPUT_DIR); print(*[x.name for x in sorted(OUTPUT_DIR.glob('*'))],sep='\n- ')